In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("ps01.ipynb")

# PS1 — Mass Action, Timescales, and Regulation Functions
### BioE 147/247 · Fall 2026

**Out:** Thursday, September 3 · **Due:** Thursday, September 10, 11:59 pm
**Covers:** Sessions 2, 3, 4 · **34 points** · BioE 247 also does Q5 (39 total)

---

Four problems. Two ask you to *derive* something and write it out; two ask you
to compute. That mix is deliberate — it is what the exams look like too.

**Collaboration is encouraged.** Discuss, argue, work at a whiteboard together,
then write your own solution and your own code. Record who you worked with
below. This is bookkeeping, not a penalty.

**If you used an LLM**, say so briefly and say what for. The conditions are
that you can explain anything you submit and that the code you submit runs.

**A note on the visible tests.** The checks you can run yourself verify
*properties* — shapes, conservation laws, scaling. They do not verify that your
answer is numerically correct; hidden tests do that on submission. A green
visible check means "not obviously broken," not "right." 

In [ ]:
COLLABORATORS = ""   # e.g. "worked with J. Chen on Q2"
AI_USE = ""          # e.g. "used an LLM to debug the Q3c loop"

## Setup

In [ ]:
# ---------------------------------------------------------------------------
# SETUP — run this cell first, every time.
#
# DataHub / local : finds the repository root and puts it on the import path.
# Google Colab    : clones the repository first, because Colab opens this
#                   notebook on its own, without the posb package beside it.
# ---------------------------------------------------------------------------
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.exists("posb2026"):
        !git clone -q https://github.com/ArkinLaboratory/posb2026.git
    sys.path.insert(0, os.path.abspath("posb2026"))
else:
    _d = os.getcwd()
    while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "posb")):
        _d = os.path.dirname(_d)
    sys.path.insert(0, _d)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import posb
from posb import Reaction, Model

posb.check_environment()


---
## Question 1 — The cell as a physical substrate

*Order-of-magnitude estimation; diffusion timescales.*

An *E. coli* cell is roughly 1 µm long and 0.5 µm across, so its volume is about
**1 femtolitre = 1 µm³**.

**Q1a.** Write `molecules(conc_nM, volume_um3)` returning the number of
molecules corresponding to a concentration in nanomolar in a volume given in
µm³.

Avogadro's number is 6.022 × 10²³ mol⁻¹ and 1 µm³ = 10⁻¹⁵ L.

In [ ]:
def molecules(conc_nM, volume_um3):
    """Number of molecules at conc_nM nanomolar in volume_um3 cubic microns."""
    ...


print(f"1 nM in 1 um^3 = {molecules(1.0, 1.0):.3f} molecules")

In [ ]:
grader.check("q1a")

**Q1b.** Write `diffusion_time(L_um, D)` for the characteristic time to
diffuse a distance *L*, using the one-dimensional convention

$$t = \frac{L^2}{2D}$$

with *L* in µm and *D* in µm²/s. **State a convention and keep it** — the factor
of 2 differs between textbooks, and a result is meaningless without it.

A typical cytoplasmic protein has *D* ≈ 10 µm²/s.

In [ ]:
def diffusion_time(L_um, D):
    """Characteristic 1-D diffusion time in seconds over L_um microns."""
    ...


for label, L in [("across E. coli (1 um)", 1.0),
                 ("across a yeast cell (5 um)", 5.0),
                 ("across a large animal cell (50 um)", 50.0)]:
    print(f"{label:<36} {diffusion_time(L, 10.0):9.4f} s")

In [ ]:
grader.check("q1b")

<!-- BEGIN QUESTION -->

**Q1c.** Transcription elongates at roughly 50 nucleotides/second, so a 1 kb
gene takes about 20 s. *E. coli* divides in roughly 30 minutes.

Using your Q1b numbers, order these three timescales — protein diffusion across
the cell, transcription of one gene, cell division — and say in **two or three
sentences** what that ordering implies about whether we need to model spatial
gradients in *E. coli*. Then say what changes for a cell 50 µm across.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 2 — From a reaction list to a model

*Mass-action rate laws; the stoichiometric matrix; numerical integration.*

$$\mathrm{E} + \mathrm{S} \;\underset{k_{-1}}{\overset{k_1}{\rightleftharpoons}}\; \mathrm{ES} \;\xrightarrow{k_2}\; \mathrm{E} + \mathrm{P}$$

<!-- BEGIN QUESTION -->

**Q2a.** Write out $d[\mathrm{E}]/dt$, $d[\mathrm{S}]/dt$,
$d[\mathrm{ES}]/dt$ and $d[\mathrm{P}]/dt$ explicitly as mass-action
expressions in $k_1, k_{-1}, k_2$.

Then state the two conservation laws this system obeys and say in one sentence
why they exist.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Q2b.** Build the stoichiometric matrix **by hand** as `S_hand`, with rows
in alphabetical order `["E", "ES", "P", "S"]` (what `posb.Model` uses by
default) and columns ordered binding, unbinding, catalysis.

Do this before running anything — the point is that you can.

In [ ]:
S_hand = ...
print(S_hand)

In [ ]:
grader.check("q2b")

**Q2c.** Build the same system as `enzyme_model` with `posb.Model`, using
parameter names `k1`, `km1`, `k2`. Then write
`product_at(t_end, params, E0, S0)` returning $[\mathrm{P}]$ at `t_end`,
starting from free enzyme `E0`, free substrate `S0`, and no ES or P.

In [ ]:
enzyme_model = ...

print(enzyme_model.summary())
print("\nmatches your hand-built S:",
      np.array_equal(np.asarray(S_hand), enzyme_model.S))

In [ ]:
def product_at(t_end, params, E0, S0):
    """[P] at time t_end from E0 free enzyme and S0 free substrate."""
    ...


p = {"k1": 1.0, "km1": 1.0, "k2": 0.1}
print(f"P(100) = {product_at(100.0, p, E0=0.01, S0=1.0):.6f}")

In [ ]:
grader.check("q2c")

---
## Question 3 — Michaelis–Menten, and where it fails

*Quasi-steady-state approximation; recognising when a reduction breaks.*

<!-- BEGIN QUESTION -->

**Q3a.** Derive the Michaelis–Menten rate law from the mechanism in Q2.

Show the steps: apply the QSSA to $[\mathrm{ES}]$, use enzyme conservation, and
arrive at

$$v = \frac{d[\mathrm{P}]}{dt} = \frac{V_{\max}[\mathrm{S}]}{K_M + [\mathrm{S}]}$$

giving $V_{\max}$ and $K_M$ in terms of $k_1, k_{-1}, k_2, E_{\text{tot}}$.

**State explicitly what the QSSA assumes** — not merely "ES is at steady state,"
but what must be true of the concentrations for that to hold.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Q3b.** Implement `vmax_km(params, E_tot)` returning the tuple
$(V_{\max}, K_M)$ from a dict with keys `k1`, `km1`, `k2`.

In [ ]:
def vmax_km(params, E_tot):
    """Return (Vmax, KM) for the Michaelis-Menten reduction."""
    ...


print(vmax_km({"k1": 1.0, "km1": 1.0, "k2": 0.1}, E_tot=0.01))

In [ ]:
grader.check("q3b")

**Q3c.** Test the approximation numerically.

Write `qssa_error(params, E0, S0, t_end)` that simulates the **full** four-species
model and the **reduced** model $dS/dt = -v$, $dP/dt = +v$ with
$v = V_{\max}S/(K_M+S)$, then returns the maximum absolute difference in
$[\mathrm{P}]$ divided by `S0`.

Then look at the two regimes printed below.

In [ ]:
def qssa_error(params, E0, S0, t_end):
    """Max |P_full - P_reduced| / S0 over [0, t_end]."""
    ...


p = {"k1": 1.0, "km1": 1.0, "k2": 0.1}
print(f"E0 = 0.001  (E0 << S0 + KM):  relative error {qssa_error(p, 0.001, 1.0, 2000):.2e}")
print(f"E0 = 1.000  (E0 ~  S0)     :  relative error {qssa_error(p, 1.000, 1.0,  200):.2e}")

In [ ]:
grader.check("q3c")

<!-- BEGIN QUESTION -->

**Q3d.** In **two or three sentences**, connect the numbers you just
computed to the condition you stated in Q3a. Which regime failed, by how much,
and does it match what your derivation predicted?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 4 — The Hill function

*Deriving a regulation function from cooperative equilibrium binding.*

<!-- BEGIN QUESTION -->

**Q4a.** A transcription factor X binds a promoter with $n$ identical
sites. Assume **strong cooperativity**: the promoter is either empty or fully
occupied, with no partially bound states populated:

$$\mathrm{P} + n\,\mathrm{X} \rightleftharpoons \mathrm{PX}_n, \qquad
K_d = \frac{[\mathrm{P}][\mathrm{X}]^n}{[\mathrm{PX}_n]}$$

Derive the bound fraction and show it takes the Hill form

$$f_{\text{bound}} = \frac{[\mathrm{X}]^n}{K^n + [\mathrm{X}]^n}$$

identifying $K$ in terms of $K_d$. Then state in one sentence each: what the
"no partially bound states" idealisation assumes, and what $n$ means
physically.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

**Q4b.** Write `hill(x, K, n)` and `sensitivity(n)`, where the sensitivity
is the fold-change in $x$ needed to go from 10% to 90% occupancy,
$x_{90}/x_{10}$.

Solve for the sensitivity **analytically** — no numerical search. You should
find it depends only on $n$.

In [ ]:
def hill(x, K, n):
    """Hill function x^n / (K^n + x^n)."""
    ...


def sensitivity(n):
    """x_90 / x_10 for a Hill function of coefficient n."""
    ...


for n in (1, 2, 4, 8):
    print(f"n = {n}:  {sensitivity(n):6.2f}-fold change in x takes you from 10% to 90%")

In [ ]:
grader.check("q4b")

---
## Question 5 — BioE 247 only

*Students in 147 may attempt this for no credit.*

<!-- BEGIN QUESTION -->

**Q5.** Q4a assumed infinitely strong cooperativity. Relax it.

Take a promoter with $n = 2$ **independent, identical** sites — binding at one
does not affect the other — each with dissociation constant $K_d$. Derive
$f_{\text{bound}}$ by writing the full partition function over the four states
(empty, site 1 only, site 2 only, both).

Show the result is a Hill function with $n = 1$, **not** $n = 2$. Then explain
what that implies for interpreting a measured Hill coefficient, and state what
would have to be true of the binding energetics to recover $n = 2$.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Submit

1. `Kernel → Restart Kernel and Run All Cells`
2. Confirm it runs top to bottom with no errors
3. Check that `COLLABORATORS` and `AI_USE` are filled in
4. Submit this `.ipynb` to **Gradescope**

Worked solutions are published after the deadline.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(run_tests=True)